# ProtSpace — Embedding Annotation Transfer (EAT)

This notebook demonstrates **Embedding Annotation Transfer (EAT)** with `protspace transfer`.
For each query protein that lacks an annotation value, the command finds the closest
annotated reference protein in pLM embedding space and transfers its label, together
with a reliability index (`confidence = 0.5 / (0.5 + distance)`, range [0, 1]).
The method follows the goPredSim approach introduced in:

- Littmann et al., *Sci Rep* 2021 — [DOI 10.1038/s41598-020-80786-0](https://doi.org/10.1038/s41598-020-80786-0)
- Heinzinger et al., *NAR Genom Bioinform* 2022 — [DOI 10.1093/nargab/lqac043](https://doi.org/10.1093/nargab/lqac043)

Distances are computed in the original high-dimensional embedding space (HDF5),
not in any 2-D/3-D projection. The curated source column is left untouched;
results are written as `COL__pred_value`, `COL__pred_confidence`, and `COL__pred_source`
columns in the bundle's annotations table.

📚 [GitHub](https://github.com/tsenoner/protspace) · [CLI Reference](https://github.com/tsenoner/protspace/blob/main/docs/cli.md#protspace-transfer) · [Annotation Reference](https://github.com/tsenoner/protspace/blob/main/docs/annotations.md#prediction-overlay-columns-eat-transfer)

## Install

In [ ]:
!pip install protspace

## Run transfer

Transfer the `protein_category` annotation from annotated reference proteins
(filtered to those with `protein_category` containing `neurotoxin`) to
unannotated query proteins whose IDs start with `TRINITY_`.

Adjust `-b`, `-e`, `-t`, `--query-id-prefix`, and `--reference-where` to match your data.

In [ ]:
!protspace transfer \
  -b results.parquetbundle \
  -e embeddings.h5:prot_t5 \
  -t protein_category \
  -o results.parquetbundle \
  --query-id-prefix TRINITY_ \
  --reference-where 'protein_category~neurotoxin'

## Read predictions back

Load the updated bundle and inspect the `*__pred_*` columns for the transferred annotations.

In [ ]:
import io
import pyarrow.parquet as pq
from protspace.data.io.bundle import read_bundle

parts, _ = read_bundle("results.parquetbundle")
df = pq.read_table(io.BytesIO(parts[0])).to_pandas()

pred_cols = [c for c in df.columns if c.endswith("__pred_value")]
df[df[pred_cols[0]].notna()].head() if pred_cols else df.head()